In [5]:
import numpy as np

class ElasticNet:

    def __init__(
        self,
        alpha=1.0,
        l1_ratio=0.5,
        epochs=1000,
        tol=1e-4
    ):
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        self.epochs = epochs
        self.tol = tol

        self.weights = None
        self.intercept = None

    def _soft_threshold(self, rho, threshold):


        if rho > threshold:
            return rho - threshold


        elif rho < -threshold:
            return rho + threshold

        else:
            return 0.0

    def fit(self, X, y):

        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).ravel()

        m, n = X.shape


        self.X_mean = X.mean(axis=0)
        self.y_mean = y.mean()

        X_centered = X - self.X_mean
        y_centered = y - self.y_mean


        self.weights = np.zeros(n)
        z = np.mean(X_centered ** 2, axis=0)


        l1_penalty = self.alpha * self.l1_ratio
        l2_penalty = self.alpha * (1 - self.l1_ratio)


        for epoch in range(self.epochs):

            old_weights = self.weights.copy()

            for j in range(n):


                if z[j] == 0:
                    self.weights[j] = 0.0
                    continue


                residual = (
                    y_centered
                    - X_centered @ self.weights
                    + X_centered[:, j] * self.weights[j]
                )

                rho = np.mean(X_centered[:, j] * residual)


                self.weights[j] = (
                    self._soft_threshold(rho, l1_penalty)
                    / (z[j] + l2_penalty)
                )

            change = np.max(
                np.abs(self.weights - old_weights)
            )

            if change < self.tol:
                break

 
        self.intercept = (
            self.y_mean - self.X_mean @ self.weights
        )

        return self

    def predict(self, X):

        X = np.asarray(X, dtype=float)

        return X @ self.weights + self.intercept

In [6]:
import pandas as pd

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet as SklearnElasticNet
from sklearn.metrics import r2_score, mean_squared_error


data = load_diabetes()
X = data.data
y = data.target


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
alpha = 1.0
l1_ratio = 0.5

my_en = ElasticNet(
    alpha=alpha,
    l1_ratio=l1_ratio,
    epochs=10000,
    tol=1e-8
)
my_en.fit(X_train_scaled, y_train)
y_pred_custom = my_en.predict(X_test_scaled)

sk_en = SklearnElasticNet(
    alpha=alpha,
    l1_ratio=l1_ratio,
    max_iter=100000,
    tol=1e-8
)

sk_en.fit(X_train_scaled, y_train)

y_pred_sklearn = sk_en.predict(X_test_scaled)

In [9]:
results = pd.DataFrame({
    "Model": [
        "Elastic Net From Scratch",
        "Sklearn Elastic Net"
    ],
    "R2 Score": [
        r2_score(y_test, y_pred_custom),
        r2_score(y_test, y_pred_sklearn)
    ],
    "MSE": [
        mean_squared_error(y_test, y_pred_custom),
        mean_squared_error(y_test, y_pred_sklearn)
    ]
})

results

,Model,R2 Score,MSE
0,Elastic Net From Scratch,0.45477,2888.711824
1,Sklearn Elastic Net,0.45477,2888.711825
